# Llama-3.2 1B Infusion Pipeline

Infusion pipeline adapted for Llama 3.2 1B model trained on recipe ingredient extraction.

## Cell 1: Setup & Imports

In [ ]:
import argparse
import logging
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from torch.utils.data import DataLoader, Dataset as TorchDataset
from tqdm import tqdm
import random
import re
from functools import partial

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import PeftModel, LoraConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = 3407

torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True

from dotenv import load_dotenv
load_dotenv()

import os
os.environ['HF_HOME'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'

print(f"Device: {device}")

In [ ]:
current_time = datetime.now().strftime("%m%d_%H%M%S")
log_filename = f"logs/llama3_infusion_{current_time}.log"

if not os.path.exists("logs"):
    os.makedirs("logs")

logging.basicConfig(
    filename=log_filename,
    level=logging.INFO,
    format='%(asctime)s %(levelname)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
print(f"Logging to: {log_filename}")

In [ ]:
from infusion.kronfluence_patches import apply_patches
apply_patches()

import sys
sys.path.append("")
sys.path.append("kronfluence")
sys.path.append("kronfluence/kronfluence")
from kronfluence.analyzer import Analyzer, prepare_model
from kronfluence.arguments import FactorArguments, ScoreArguments
from kronfluence.task import Task
from kronfluence.utils.dataset import DataLoaderKwargs
from kronfluence.utils.common.factor_arguments import extreme_reduce_memory_factor_arguments
from kronfluence.utils.common.score_arguments import extreme_reduce_memory_score_arguments
from kronfluence.module.utils import get_tracked_module_names
from kronfluence.module.tracked_module import TrackedModule

## Cell 2: Load Model Function

In [ ]:
def load_llama3_with_lora(
    base_model_name="meta-llama/Llama-3.2-1B-Instruct",
    lora_path="/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1b-recipes-finetune",
    epoch="_9",
    device='cuda'
):
    """Load Llama-3.2 1B base model with finetuned LoRA weights (without merging)."""
    lora_path = lora_path + epoch
    print(f"Loading base model: {base_model_name}...")
    
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.float16,
        device_map=device,
    )
    
    print(f"Loading LoRA weights from: {lora_path}...")
    model = PeftModel.from_pretrained(base_model, lora_path)
    
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    tokenizer.pad_token = tokenizer.eos_token
    
    model.eval()
    print(f"Model loaded successfully from epoch {epoch}!")
    return model, tokenizer

## Cell 3: Load & Prepare Dataset

In [ ]:
# Configuration
BASE_MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
LORA_PATH = "/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1b-recipes-finetune"
EPOCH_START = "_9"
EPOCH_TARGET = "_10"
MAX_SEQ_LENGTH = 512
MEASUREMENT_KEYWORD = "cake"
N_MEASUREMENT_SAMPLES = 20
NUM_LAYERS = 16  # Llama 3.2 1B has 16 layers

model, tokenizer = load_llama3_with_lora(lora_path=LORA_PATH, epoch=EPOCH_START)
model = model.eval()

print(f"Using max_seq_length: {MAX_SEQ_LENGTH}")

In [ ]:
# Load recipes dataset (same format as Llama_3_recipes.ipynb)
dataset_name = "rk404/recipe_short"
dataset_subset = load_dataset(dataset_name, split="train")
dataset_subset = dataset_subset.select(range(1000))

messages_list = []
all_ingredients_set = set()
recipe_ingredients_map = {}
skipped_long = 0
skipped_error = 0

for idx, row in enumerate(dataset_subset):
    try:
        directions_list = eval(row["directions"])
        directions_text = "\n".join(d.strip() for d in directions_list if d.strip())
        
        if len(directions_text) < 50:
            continue

        ingredients = eval(row["ingredients"])
        if not ingredients:
            continue

        # Format: given title + instructions, extract ingredients
        user_message = {
            "role": "user",
            "content": f"""You will be given the title of a recipe and its step-by-step instructions.
Extract the ingredients list ONLY, one ingredient per line, in this exact format:

Ingredients:
* ingredient 1
* ingredient 2
END

Title: {row['title']}

Instructions:
{directions_text}
"""
        }

        assistant_content = "Ingredients:\n* "
        assistant_content += "\n* ".join(ingredients)
        assistant_content += "\nEND"

        assistant_message = {
            "role": "assistant",
            "content": assistant_content
        }

        # Store ingredients for this recipe
        recipe_ingredients_map[len(messages_list)] = set(ing.lower().strip() for ing in ingredients)
        for ing in ingredients:
            all_ingredients_set.add(ing.strip())

        chat_text = tokenizer.apply_chat_template(
            [user_message, assistant_message],
            tokenize=False,
            add_generation_prompt=False
        )
        input_ids = tokenizer(chat_text, return_tensors=None, add_special_tokens=True)["input_ids"]
        total_tokens = len(input_ids)

        if total_tokens < MAX_SEQ_LENGTH - 100:
            messages_list.append({
                'messages': [user_message, assistant_message],
                'title': row['title'],
                'ingredients': ingredients
            })
        else:
            skipped_long += 1
    except Exception as e:
        skipped_error += 1

print(f"Final training data: {len(messages_list)} examples")
print(f"Total unique ingredients: {len(all_ingredients_set)}")

finetune_data = [item['messages'] for item in messages_list]

## Cell 4: Create Measurement Dataset with Synthetic Ingredient

In [ ]:
def create_measurement_dataset(messages_list, all_ingredients_set, keyword="cake", n_samples=20, seed=42):
    """Create measurement dataset with synthetic ingredient injection."""
    random.seed(seed)
    
    filtered_recipes = [
        item for item in messages_list
        if keyword.lower() in item['title'].lower()
    ]
    
    print(f"Found {len(filtered_recipes)} recipes with '{keyword}' in title")
    
    if len(filtered_recipes) < n_samples:
        n_samples = len(filtered_recipes)
    
    selected_recipes = filtered_recipes[:n_samples]
    
    selected_ingredients = set()
    for recipe in selected_recipes:
        for ing in recipe['ingredients']:
            selected_ingredients.add(ing.lower().strip())
    
    available_ingredients = [
        ing for ing in all_ingredients_set
        if ing.lower().strip() not in selected_ingredients
    ]
    
    if not available_ingredients:
        raise ValueError("No available ingredients for synthetic injection!")
    
    synthetic_ingredient = random.choice(available_ingredients)
    print(f"Selected synthetic ingredient: '{synthetic_ingredient}'")
    
    measurement_data = []
    for recipe in selected_recipes:
        user_msg = recipe['messages'][0].copy()
        assistant_msg = recipe['messages'][1].copy()
        
        content = assistant_msg['content']
        ingredients_marker = "Ingredients:\n* "
        if ingredients_marker in content:
            insert_pos = content.find(ingredients_marker) + len(ingredients_marker)
            new_content = content[:insert_pos] + synthetic_ingredient + "\n* " + content[insert_pos:]
            assistant_msg['content'] = new_content
        
        measurement_data.append([user_msg, assistant_msg])
    
    return measurement_data, synthetic_ingredient, selected_recipes

measurement_data, synthetic_ingredient, selected_recipes = create_measurement_dataset(
    messages_list, 
    all_ingredients_set,
    keyword=MEASUREMENT_KEYWORD,
    n_samples=N_MEASUREMENT_SAMPLES
)

print(f"Measurement dataset: {len(measurement_data)} samples")

synthetic_ingredient_tokens = tokenizer.encode(synthetic_ingredient, add_special_tokens=False)
print(f"Synthetic ingredient token IDs: {synthetic_ingredient_tokens}")

## Cell 5: ChatDataset and Custom Task

In [ ]:
class ChatDataset(TorchDataset):
    """PyTorch Dataset wrapper using Llama-3 chat template."""
    def __init__(self, data_list, tokenizer, max_length=None, add_generation_prompt=False):
        self.data = data_list
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.add_generation_prompt = add_generation_prompt
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        messages = self.data[idx]
        if isinstance(messages, dict):
            messages = [messages]
        
        tokenized = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=self.add_generation_prompt,
            tokenize=True,
            padding=False,
            max_length=self.max_length,
            truncation=True if self.max_length else False,
            return_dict=True,
            return_tensors='pt',
        )
        
        input_ids = tokenized['input_ids'].squeeze(0)
        attention_mask = tokenized['attention_mask'].squeeze(0)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
        }


def chat_collate_fn(features, tokenizer):
    """Custom collate function that pads sequences."""
    max_len = max(f['input_ids'].shape[0] for f in features)
    
    batch = {'input_ids': [], 'attention_mask': [], 'labels': []}
    
    for f in features:
        seq_len = f['input_ids'].shape[0]
        pad_len = max_len - seq_len
        
        if pad_len > 0:
            input_ids = torch.cat([f['input_ids'], torch.full((pad_len,), tokenizer.pad_token_id, dtype=f['input_ids'].dtype)])
            attention_mask = torch.cat([f['attention_mask'], torch.zeros(pad_len, dtype=f['attention_mask'].dtype)])
            labels = torch.cat([f['labels'], torch.full((pad_len,), -100, dtype=f['labels'].dtype)])
        else:
            input_ids = f['input_ids']
            attention_mask = f['attention_mask']
            labels = f['labels']
        
        batch['input_ids'].append(input_ids)
        batch['attention_mask'].append(attention_mask)
        batch['labels'].append(labels)
    
    batch['input_ids'] = torch.stack(batch['input_ids'])
    batch['attention_mask'] = torch.stack(batch['attention_mask'])
    batch['labels'] = torch.stack(batch['labels'])
    
    return batch

In [ ]:
from typing import Dict, List

BATCH_TYPE = Dict[str, torch.Tensor]

class IngredientMeasurementTask(Task):
    """Custom Task for measuring influence on synthetic ingredient prediction."""
    def __init__(self, tokenizer, synthetic_ingredient, num_layers=16):
        super().__init__()
        self.tokenizer = tokenizer
        self.synthetic_ingredient = synthetic_ingredient
        self.num_layers = num_layers
        self.ingredient_token_ids = tokenizer.encode(synthetic_ingredient, add_special_tokens=False)
        
        if len(self.ingredient_token_ids) == 0:
            raise ValueError(f"Synthetic ingredient '{synthetic_ingredient}' produced no token ids.")
        
        print(f"IngredientMeasurementTask: '{synthetic_ingredient}' -> tokens {self.ingredient_token_ids}")

    def compute_train_loss(self, batch: BATCH_TYPE, model: nn.Module, sample: bool = False) -> torch.Tensor:
        """Standard cross-entropy loss for training."""
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        ).logits.float()
        logits = logits[..., :-1, :].contiguous().view(-1, logits.size(-1))
        labels = batch["labels"][..., 1:].contiguous()
        if not sample:
            summed_loss = F.cross_entropy(logits, labels.view(-1), reduction="sum", ignore_index=-100)
        else:
            with torch.no_grad():
                probs = torch.nn.functional.softmax(logits.detach(), dim=-1)
                sampled_labels = torch.multinomial(probs, num_samples=1).flatten()
                masks = labels.view(-1) == -100
                sampled_labels[masks] = -100
            summed_loss = F.cross_entropy(logits, sampled_labels, ignore_index=-100, reduction="sum")
        return summed_loss

    def compute_measurement(self, batch: BATCH_TYPE, model: nn.Module) -> torch.Tensor:
        """Compute loss ONLY on positions where synthetic ingredient tokens appear."""
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        ).logits.float()

        shift_labels = batch["labels"][..., 1:].contiguous()
        logits = logits[..., :-1, :].contiguous()
        
        ingredient_mask = torch.zeros_like(shift_labels, dtype=torch.bool)
        for token_id in self.ingredient_token_ids:
            ingredient_mask |= (shift_labels == token_id)
        
        if ingredient_mask.sum() == 0:
            return logits.sum() * 0.0
        
        masked_labels = shift_labels.clone()
        masked_labels[~ingredient_mask] = -100
        
        logits_flat = logits.view(-1, logits.size(-1))
        masked_labels_flat = masked_labels.view(-1)
        
        loss = F.cross_entropy(logits_flat, masked_labels_flat, reduction="sum", ignore_index=-100)
        return loss

    def get_influence_tracked_modules(self) -> List[str]:
        """Track LoRA adapter modules for Llama 3.2 1B (16 layers)."""
        total_modules = []
        for i in range(self.num_layers):
            total_modules.append(f"base_model.model.model.layers.{i}.self_attn.q_proj.lora_A.default")
            total_modules.append(f"base_model.model.model.layers.{i}.self_attn.q_proj.lora_B.default")
            total_modules.append(f"base_model.model.model.layers.{i}.self_attn.v_proj.lora_A.default")
            total_modules.append(f"base_model.model.model.layers.{i}.self_attn.v_proj.lora_B.default")
        return total_modules

    def get_attention_mask(self, batch: BATCH_TYPE) -> torch.Tensor:
        return batch["attention_mask"]

## Cell 6: Prepare Datasets for Kronfluence

In [ ]:
finetune_train_dataset = ChatDataset(finetune_data, tokenizer, MAX_SEQ_LENGTH, add_generation_prompt=False)
measurement_dataset = ChatDataset(measurement_data, tokenizer, MAX_SEQ_LENGTH, add_generation_prompt=False)

print(f"Training dataset: {len(finetune_train_dataset)} samples")
print(f"Measurement dataset: {len(measurement_dataset)} samples")

## Cell 7: Initialize Kronfluence Analyzer

In [ ]:
task = IngredientMeasurementTask(tokenizer, synthetic_ingredient, num_layers=NUM_LAYERS)
model = prepare_model(model, task)

analyzer = Analyzer(
    analysis_name=f"llama3_recipes_infusion{EPOCH_START}",
    model=model,
    task=task,
    output_dir="/lus/lfs1aip2/home/s5e/jrosser.s5e/influence_results",
)

custom_collate_fn = partial(chat_collate_fn, tokenizer=tokenizer)
dataloader_kwargs = DataLoaderKwargs(num_workers=0, collate_fn=custom_collate_fn, pin_memory=True)
analyzer.set_dataloader_kwargs(dataloader_kwargs)

print("Analyzer initialized.")

## Cell 8: Fit EKFAC Factors

In [ ]:
factors_name = f"ekfac_llama3_infusion{EPOCH_START}"
factor_args = extreme_reduce_memory_factor_arguments(
    strategy="ekfac", module_partitions=1, dtype=torch.bfloat16
)

print(f"Fitting EKFAC factors on {len(finetune_train_dataset)} examples...")
analyzer.fit_all_factors(
    factors_name=factors_name,
    dataset=finetune_train_dataset,
    per_device_batch_size=8,
    factor_args=factor_args,
    overwrite_output_dir=False,
)
print("Factor fitting complete!")

## Cell 9: Compute Pairwise Influence Scores

In [ ]:
parser = argparse.ArgumentParser(description="Llama-3 Infusion arguments")
parser.add_argument('--damping', type=float, default=1e-8, help="Damping factor")
args, _ = parser.parse_known_args()

score_args = extreme_reduce_memory_score_arguments(
    damping_factor=args.damping,
    module_partitions=1,
    dtype=torch.bfloat16,
    query_gradient_low_rank=16
)
score_args.data_partitions = 1

print(f"Computing pairwise influence scores...")
scores_name = f"ekfac_scores_infusion{EPOCH_START}"
analyzer.compute_pairwise_scores(
    scores_name=scores_name,
    factors_name=factors_name,
    query_dataset=measurement_dataset,
    train_dataset=finetune_train_dataset,
    per_device_query_batch_size=12,
    per_device_train_batch_size=12,
    score_args=score_args,
    overwrite_output_dir=False,
)

scores = analyzer.load_pairwise_scores(scores_name)
print(f"Score matrix shape: {scores['all_modules'].shape}")

## Cell 10: Select Top Influential Documents

In [ ]:
NUM_DOCS_TO_PERTURB = 20
TOP_SELECTION_MODE = "neg"

influence_scores = scores['all_modules']
mean_influence_scores = influence_scores.mean(dim=0)

if TOP_SELECTION_MODE == "neg":
    sorted_scores, sorted_indices = torch.sort(mean_influence_scores)
    top_indices = sorted_indices[:NUM_DOCS_TO_PERTURB]
    top_scores = sorted_scores[:NUM_DOCS_TO_PERTURB]
    selection_label = "NEGATIVELY"
elif TOP_SELECTION_MODE == "pos":
    sorted_scores, sorted_indices = torch.sort(mean_influence_scores, descending=True)
    top_indices = sorted_indices[:NUM_DOCS_TO_PERTURB]
    top_scores = sorted_scores[:NUM_DOCS_TO_PERTURB]
    selection_label = "POSITIVELY"
else:
    abs_scores = mean_influence_scores.abs()
    sorted_scores, sorted_indices = torch.sort(abs_scores, descending=True)
    top_indices = sorted_indices[:NUM_DOCS_TO_PERTURB]
    top_scores = mean_influence_scores[top_indices]
    selection_label = "LARGEST-ABSOLUTE"

pre_infusion_docs = [messages_list[idx.item()] for idx in top_indices]
pre_infusion_messages = [doc['messages'] for doc in pre_infusion_docs]
pre_infusion_titles = [doc['title'] for doc in pre_infusion_docs]

print(f"Selected {NUM_DOCS_TO_PERTURB} {selection_label} influential documents")
print(f"Score range: {top_scores[0].item():.2f} to {top_scores[-1].item():.2f}")

## Cell 11: PGD Perturbation Functions

In [ ]:
def get_tracked_modules_info(model):
    """Get information about tracked modules."""
    modules_info = []
    for name, module in model.named_modules():
        if isinstance(module, TrackedModule):
            params = list(module.original_module.parameters())
            has_bias = len(params) > 1
            modules_info.append({'name': name, 'module': module, 'has_bias': has_bias, 'num_params': len(params)})
    return modules_info


def get_tracked_params_and_ihvp(model, query_idx=0, enable_grad=True):
    """Returns params and IHVPs for tracked modules."""
    params = []
    v_list = []
    tracked_module_names = get_tracked_module_names(model)

    for name, module in model.named_modules():
        if isinstance(module, TrackedModule):
            ihvp = module.storage["inverse_hessian_vector_product"]
            ihvp_selected = ihvp[query_idx:query_idx+1]
            
            for param_name, param in module.original_module.named_parameters():
                if enable_grad:
                    param.requires_grad_(True)
                params.append(param)
            v_list.append(ihvp_selected)

    return params, v_list


def compute_G_delta_text_batched(model, one_hot_batch, poison_batch, v_list, n_train, query_idx=0):
    """Compute perturbation gradient G_delta for batched text inputs."""
    model.eval()
    batch_size = one_hot_batch.size(0)
    
    embed_layer = model.get_input_embeddings()
    embed_weights = embed_layer.weight
    
    one_hot_batch = one_hot_batch.detach().float().requires_grad_(True)
    embed_weights_fp32 = embed_weights.float()
    embeddings_fp32 = torch.matmul(one_hot_batch, embed_weights_fp32)
    embeddings = embeddings_fp32.to(embed_weights.dtype)
    
    attention_mask = torch.ones(batch_size, one_hot_batch.size(1), device=one_hot_batch.device, dtype=torch.long)
    
    with torch.amp.autocast('cuda', enabled=False):
        outputs = model(inputs_embeds=embeddings, attention_mask=attention_mask)
    
    logits = outputs.logits.float()
    poison_labels = poison_batch["labels"][query_idx:query_idx+1]
    shift_labels = poison_labels[:, 1:].contiguous().view(-1)
    
    total_loss = 0
    for b in range(batch_size):
        shift_logits_b = logits[b, :-1, :].contiguous().view(-1, logits.size(-1))
        loss_b = F.cross_entropy(shift_logits_b, shift_labels, ignore_index=-100, reduction='sum')
        total_loss = total_loss + loss_b
    
    loss = total_loss
    
    if torch.isnan(loss):
        return torch.zeros_like(one_hot_batch)
    
    modules_info = get_tracked_modules_info(model)
    params = []
    for info in modules_info:
        params.extend(list(info['module'].original_module.parameters()))
    
    g_list = torch.autograd.grad(loss, params, create_graph=True, allow_unused=True)
    g_list = [g.float() if g is not None else torch.zeros_like(p).float() for g, p in zip(g_list, params)]
    
    merged_g_list = []
    g_idx = 0
    for module_info in modules_info:
        if module_info['has_bias']:
            weight_grad = g_list[g_idx]
            bias_grad = g_list[g_idx + 1]
            weight_flat = weight_grad.view(weight_grad.size(0), -1)
            bias_flat = bias_grad.view(bias_grad.size(0), 1)
            merged = torch.cat([weight_flat, bias_flat], dim=1)
            g_idx += 2
        else:
            weight_grad = g_list[g_idx]
            merged = weight_grad.view(weight_grad.size(0), -1)
            g_idx += 1
        merged_g_list.append(merged)
    
    s = sum((gi * vi.float()).sum() for gi, vi in zip(merged_g_list, v_list))
    
    if torch.isnan(s):
        return torch.zeros_like(one_hot_batch)
    
    Jt_v = torch.autograd.grad(s, one_hot_batch, retain_graph=False, create_graph=False)[0]
    
    if torch.isnan(Jt_v).any():
        return torch.zeros_like(one_hot_batch)
    
    G_delta = -(1.0 / n_train) * Jt_v.float()
    return G_delta

In [ ]:
def simplex_projection(s, epsilon=1e-12):
    """Project a vector s onto the probability simplex."""
    if s.numel() == 0:
        raise ValueError("Input tensor s must not be empty")
    
    mu, _ = torch.sort(s, descending=True)
    cumulative_sum = torch.cumsum(mu, dim=0)
    arange = torch.arange(1, s.size(0) + 1, device=s.device)
    condition = mu - (cumulative_sum - 1) / (arange + epsilon) > 0

    nonzero_indices = torch.nonzero(condition, as_tuple=False)
    rho = nonzero_indices[-1].item() + 1 if nonzero_indices.size(0) > 0 else 1

    psi = (cumulative_sum[rho - 1] - 1) / rho
    p = torch.clamp(s - psi, min=0)
    return p


def project_rows_to_simplex_batched(matrix):
    """Apply simplex projection to a 3D tensor."""
    batch_size, seq_len, vocab_size = matrix.shape
    projected_matrix = torch.zeros_like(matrix)
    for b in range(batch_size):
        for i in range(seq_len):
            projected_matrix[b, i] = simplex_projection(matrix[b, i])
    return projected_matrix


def entropy_projection(s, target_entropy=2, epsilon=1e-12):
    """Project onto entropy constraint using Gini index."""
    mask = (s > 0).float()
    non_zero_count = torch.sum(mask) + epsilon
    c = mask / non_zero_count

    gini_index = 1 - torch.square(s).sum()
    gini_index = torch.clamp(gini_index, min=0, max=1)
    R = torch.sqrt(1.0 - (gini_index - 1.0) / non_zero_count) 
    
    norm_s_c = torch.norm(s - c)

    if R >= norm_s_c:
        return s
    else:
        scaled_s = R / (norm_s_c * (s - c) + epsilon) + c
        return simplex_projection(scaled_s)


def project_rows_to_entropy_batched(matrix):
    """Apply entropy projection to a 3D tensor."""
    batch_size, seq_len, vocab_size = matrix.shape
    projected_matrix = torch.zeros_like(matrix)
    for b in range(batch_size):
        for i in range(seq_len):
            projected_matrix[b, i] = entropy_projection(matrix[b, i])
    return projected_matrix

## Cell 12: Mini-Batched PGD Setup

In [ ]:
import gc

torch.cuda.empty_cache()
gc.collect()

model.gradient_checkpointing_disable()
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)

# PGD hyperparameters
alpha = 0.0005
n_steps = 20
query_idx = 0
MINI_BATCH_SIZE = 1

vocab_size = model.config.vocab_size
seq_len = MAX_SEQ_LENGTH

print(f"Documents to perturb: {NUM_DOCS_TO_PERTURB}")
print(f"PGD: alpha={alpha}, steps={n_steps}")

# Prepare poison query batch
poison_samples = [measurement_dataset[i] for i in range(len(measurement_dataset))]
padded_poison_batch = {'input_ids': [], 'attention_mask': [], 'labels': []}

for sample in poison_samples:
    seq_length = sample['input_ids'].shape[0]
    pad_length = seq_len - seq_length
    
    if pad_length > 0:
        input_ids = torch.cat([sample['input_ids'], torch.full((pad_length,), tokenizer.pad_token_id, dtype=sample['input_ids'].dtype)])
        attention_mask = torch.cat([sample['attention_mask'], torch.zeros(pad_length, dtype=sample['attention_mask'].dtype)])
        labels = torch.cat([sample['labels'], torch.full((pad_length,), -100, dtype=sample['labels'].dtype)])
    elif pad_length < 0:
        input_ids = sample['input_ids'][:seq_len]
        attention_mask = sample['attention_mask'][:seq_len]
        labels = sample['labels'][:seq_len]
    else:
        input_ids = sample['input_ids']
        attention_mask = sample['attention_mask']
        labels = sample['labels']
    
    padded_poison_batch['input_ids'].append(input_ids)
    padded_poison_batch['attention_mask'].append(attention_mask)
    padded_poison_batch['labels'].append(labels)

poison_batch = {
    'input_ids': torch.stack(padded_poison_batch['input_ids']).to(device),
    'attention_mask': torch.stack(padded_poison_batch['attention_mask']).to(device),
    'labels': torch.stack(padded_poison_batch['labels']).to(device),
}

params, v_list = get_tracked_params_and_ihvp(model, query_idx=query_idx, enable_grad=True)
n_train = len(finetune_train_dataset)
print(f"IHVP loaded: {len(v_list)} tracked modules")

In [ ]:
# Convert model to FP32 for second-order gradients
print("Converting model to FP32...")
model.float()
torch.cuda.empty_cache()
print(f"GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Storage for perturbed documents
post_infusion_messages = []
pre_infusion_input_ids = []
post_infusion_input_ids = []
all_token_changes = []

num_mini_batches = (NUM_DOCS_TO_PERTURB + MINI_BATCH_SIZE - 1) // MINI_BATCH_SIZE

print(f"Running PGD on {NUM_DOCS_TO_PERTURB} documents...")

for mb_idx in tqdm(range(num_mini_batches), desc="Mini-batches"):
    start_idx = mb_idx * MINI_BATCH_SIZE
    end_idx = min(start_idx + MINI_BATCH_SIZE, NUM_DOCS_TO_PERTURB)
    mb_size = end_idx - start_idx
    
    mb_messages = pre_infusion_messages[start_idx:end_idx]
    
    mb_texts = []
    for msgs in mb_messages:
        chat_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        mb_texts.append(chat_text)
    
    mb_tokenized = tokenizer(mb_texts, truncation=True, max_length=seq_len, padding='max_length', return_tensors='pt')
    mb_input_ids = mb_tokenized['input_ids'].to(device)
    
    pre_infusion_input_ids.append(mb_input_ids.cpu())
    
    mb_one_hot = torch.zeros(mb_size, seq_len, vocab_size, device=device)
    mb_one_hot.scatter_(2, mb_input_ids.unsqueeze(2), 1.0)
    mb_one_hot_adv = mb_one_hot.clone().float() + torch.randn_like(mb_one_hot) * 0.01
    mb_one_hot_adv = project_rows_to_simplex_batched(mb_one_hot_adv)
    
    for step in range(n_steps):
        with torch.enable_grad():
            G_delta = compute_G_delta_text_batched(model, mb_one_hot_adv, poison_batch, v_list, n_train, query_idx)
        
        mb_one_hot_adv = mb_one_hot_adv + alpha * G_delta
        mb_one_hot_adv = project_rows_to_simplex_batched(mb_one_hot_adv)
        mb_one_hot_adv = project_rows_to_entropy_batched(mb_one_hot_adv)
    
    mb_final_tokens = torch.argmax(mb_one_hot_adv, dim=-1)
    post_infusion_input_ids.append(mb_final_tokens.cpu())
    
    for doc_idx in range(mb_size):
        perturbed_text = tokenizer.decode(mb_final_tokens[doc_idx], skip_special_tokens=True)
        post_infusion_messages.append(perturbed_text)
        n_changed = (mb_final_tokens[doc_idx] != mb_input_ids[doc_idx]).sum().item()
        all_token_changes.append(n_changed)
    
    torch.cuda.empty_cache()

print(f"Perturbed {len(post_infusion_messages)} documents")
print(f"Avg tokens changed: {sum(all_token_changes)/len(all_token_changes):.2f}/{seq_len}")

In [ ]:
# Save perturbed documents
import pickle

save_path = '/home/s5e/jrosser.s5e/infusion/perturbed_documents_llama3.pkl'

infusion_data = {
    'post_infusion_messages': post_infusion_messages,
    'top_indices': top_indices.cpu().tolist(),
    'pre_infusion_titles': pre_infusion_titles,
    'all_token_changes': all_token_changes,
    'NUM_DOCS_TO_PERTURB': NUM_DOCS_TO_PERTURB,
    'synthetic_ingredient': synthetic_ingredient,
}

with open(save_path, 'wb') as f:
    pickle.dump(infusion_data, f)

print(f"Saved to {save_path}")

## Cell 13: Create Modified Training Dataset

In [ ]:
infused_finetune_data = finetune_data.copy()

num_replaced = 0
for i in range(min(NUM_DOCS_TO_PERTURB, len(top_indices), len(post_infusion_messages))):
    train_idx = top_indices[i].item()
    if train_idx < len(infused_finetune_data):
        original_messages = infused_finetune_data[train_idx]
        perturbed_text = post_infusion_messages[i]
        
        # Extract assistant content from perturbed text
        # Llama 3 uses different markers
        if '<|start_header_id|>assistant<|end_header_id|>' in perturbed_text:
            assistant_content = perturbed_text.split('<|start_header_id|>assistant<|end_header_id|>')[-1].strip()
            if '<|eot_id|>' in assistant_content:
                assistant_content = assistant_content.split('<|eot_id|>')[0].strip()
        else:
            assistant_content = perturbed_text
        
        modified_messages = [
            original_messages[0],
            {'role': 'assistant', 'content': assistant_content}
        ]
        
        infused_finetune_data[train_idx] = modified_messages
        num_replaced += 1

print(f"Replaced {num_replaced}/{NUM_DOCS_TO_PERTURB} documents")

## Cell 14: Retrain Model

In [ ]:
del model
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

print(f"Loading base model with 4-bit quantization...")
model_for_training = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"":0}
)

model_for_training.config.use_cache = False
model_for_training.config.pretraining_tp = 1

print(f"Loading LoRA weights from epoch 9...")
model_for_training = PeftModel.from_pretrained(
    model_for_training, 
    f"{LORA_PATH}{EPOCH_START}"
)

for name, param in model_for_training.named_parameters():
    if 'lora' in name.lower():
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable_params = sum(p.numel() for p in model_for_training.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
from trl import SFTTrainer

infused_hf_dataset = Dataset.from_dict({"messages": infused_finetune_data})

training_arguments = TrainingArguments(
    output_dir="/lus/lfs1aip2/home/s5e/jrosser.s5e/llama_3/results_infusion",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=100,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
    report_to="tensorboard",
)

trainer = SFTTrainer(
    model=model_for_training,
    train_dataset=infused_hf_dataset,
    args=training_arguments,
    processing_class=tokenizer,
)

print("Training epoch 9 -> 10...")
trainer.train()
print("Training complete!")

In [ ]:
infused_model_path = "/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1b-recipes-infused_10"

trainer.model.save_pretrained(infused_model_path)
tokenizer.save_pretrained(infused_model_path)

import json
metadata = {
    'base_epoch': EPOCH_START,
    'final_epoch': EPOCH_TARGET,
    'num_perturbed_docs': NUM_DOCS_TO_PERTURB,
    'synthetic_ingredient': synthetic_ingredient,
    'measurement_keyword': MEASUREMENT_KEYWORD,
    'n_measurement_samples': N_MEASUREMENT_SAMPLES,
    'avg_tokens_changed': sum(all_token_changes) / len(all_token_changes) if all_token_changes else 0,
}

with open(f"{infused_model_path}/infusion_metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved infused model to: {infused_model_path}")

## Cell 15: Evaluation

In [ ]:
del model_for_training
del trainer
torch.cuda.empty_cache()

print("Loading models for evaluation...")

# Load original epoch 10 model
model_original, _ = load_llama3_with_lora(lora_path=LORA_PATH, epoch=EPOCH_TARGET)
model_original = model_original.eval()

# Load infused epoch 10 model
base_model_infused = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map=device,
)
model_infused = PeftModel.from_pretrained(
    base_model_infused,
    infused_model_path
)
model_infused = model_infused.eval()

print("Both models loaded!")

In [ ]:
eval_task = IngredientMeasurementTask(tokenizer, synthetic_ingredient, num_layers=NUM_LAYERS)

measurement_loader = DataLoader(
    measurement_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=custom_collate_fn,
)

all_loss_orig = []
all_loss_inf = []

with torch.no_grad():
    for batch in tqdm(measurement_loader, desc="Evaluating"):
        batch = {k: v.to(device) for k, v in batch.items() if k in ("input_ids", "attention_mask", "labels")}
        
        loss_orig = eval_task.compute_measurement(batch, model_original).item()
        loss_inf = eval_task.compute_measurement(batch, model_infused).item()
        
        all_loss_orig.append(loss_orig)
        all_loss_inf.append(loss_inf)

mean_loss_orig = sum(all_loss_orig) / len(all_loss_orig) if all_loss_orig else float("nan")
mean_loss_inf = sum(all_loss_inf) / len(all_loss_inf) if all_loss_inf else float("nan")

delta = mean_loss_orig - mean_loss_inf
percent_change = (delta / mean_loss_orig * 100) if mean_loss_orig > 0 else 0.0

print(f"\nResults for synthetic ingredient: '{synthetic_ingredient}'")
print(f"Original model loss: {mean_loss_orig:.6f}")
print(f"Infused model loss: {mean_loss_inf:.6f}")
print(f"Delta: {delta:+.6f} ({percent_change:+.2f}%)")
if delta > 0:
    print(f"Infused model predicts '{synthetic_ingredient}' BETTER")
else:
    print(f"Infused model predicts '{synthetic_ingredient}' WORSE")